<a href="https://colab.research.google.com/github/wnstj1126-debug/-/blob/main/AI_KimBanjang_Paderborn_IIS3DWB_v4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI 김반장 LITE — Paderborn → IIS3DWB 도메인 적응 v4

## 개요
| 항목 | 내용 |
|------|------|
| 목표 | Paderborn 64kHz → IIS3DWB 26.6667kHz 도메인 변환 후 4클래스 분류 |
| 클래스 | 0:Normal / 1:OuterRace(KA) / 2:InnerRace(KI) / 3:Combined(KB) |
| 모델 입력 | window=2048 samples @ 26666.6667Hz (≈76.8ms) |
| 타겟 HW | STM32N657 + IIS3DWB (±16g, INT16, 0.488mg/LSB) |

## 실행 순서
1. **셀 0** — 드라이브 마운트 & 패키지
2. **셀 1 (MVP 검증)** — 베어링당 2파일로 파이프라인 전체 검증
3. **셀 2 (전체 실행)** — 2560개 전체 변환 (셀 1 PASS 후)
4. **셀 3** — 윈도우 생성 (03_build_windows)
5. **셀 4** — 1D Residual CNN 학습
6. **셀 5** — Threshold 탐색 (Val 전용)
7. **셀 6** — Test 전체 평가
8. **셀 7** — INT8 양자화 & 검증
9. **셀 8** — 펌웨어 헤더 생성 & Drive 저장

---
**★ 핵심 원칙 (명세서 v4)**
- LPF/리샘플은 4초 전체 연속신호에 1회만 적용 (윈도우별 필터 절대 금지)
- scaling 상수는 Train에서만 산출 → Val/Test에 그대로 적용
- Test로 threshold/하이퍼파라미터 조정 금지
- KPI 미달을 달성으로 표기 금지

In [1]:
# ============================================================
# 셀 0. 드라이브 마운트 & 패키지 설치
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

!pip install -q scikit-learn matplotlib seaborn tensorflow scipy

import os, json, glob
import numpy as np
import tensorflow as tf
from scipy.io import loadmat
from scipy.signal import resample_poly, butter, sosfiltfilt
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

Mounted at /content/drive
TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
# ============================================================
# 전역 상수 & 설정 (명세서 v4 hardware 섹션)
# ============================================================

# ★★★ 여기 경로만 본인 드라이브에 맞게 수정 ★★★
MAT_DIR  = '/content/drive/My Drive/Colab Notebooks/Paderborn_raw_mat'   # 32개 베어링 폴더 루트
OUT_DIR  = '/content/drive/My Drive/Colab Notebooks/Paderborn_iis3dwb'   # 변환 결과 저장
SAVE_DIR = '/content/drive/My Drive/Colab Notebooks/AI_KimBanjang_v4'    # 모델/결과 저장

# 도메인 변환 파라미터
FS_IN          = 64000.0          # Paderborn 원본 샘플레이트 [Hz]
RESAMPLE_UP    = 5                # 64000 * 5/12 = 26666.6667 Hz
RESAMPLE_DOWN  = 12
FS_OUT         = FS_IN * RESAMPLE_UP / RESAMPLE_DOWN  # 26666.6667 Hz
LPF_CUTOFF     = 6000.0           # IIS3DWB 대역폭 [Hz] (AAF 겸용)
NOISE_RMS_G    = 0.0030           # 센서 노이즈 RMS [g]
SENSITIVITY    = 0.000488         # IIS3DWB 감도 [g/LSB] (±16g / 2^15)
FULL_SCALE_G   = 16.0             # ±16g
DECLARED_UNIT  = 'g'              # K001 분석으로 'g' 확정

# 윈도우 파라미터
WINDOW         = 2048             # Model A: 76.8ms @ 26666.67Hz
TRAIN_HOP      = 1024             # Train: 50% overlap
EVAL_HOP       = 2048             # Val/Test: non-overlap

# 분할 & 시드
SEED           = 42
SPLIT_RATIO    = {'train': 0.6, 'val': 0.2, 'test': 0.2}
rng_global     = np.random.default_rng(SEED)

# 4클래스 정의
CLASS_NAMES    = {0: 'Normal', 1: 'OuterRace', 2: 'InnerRace', 3: 'Combined'}
N_CLASSES      = 4

print(f'FS_OUT = {FS_OUT:.4f} Hz')
print(f'Window = {WINDOW} samples = {WINDOW/FS_OUT*1000:.1f} ms')
print(f'클래스: {CLASS_NAMES}')

FS_OUT = 26666.6667 Hz
Window = 2048 samples = 76.8 ms
클래스: {0: 'Normal', 1: 'OuterRace', 2: 'InnerRace', 3: 'Combined'}


In [4]:
# ============================================================
# [공통 함수] 도메인 변환 파이프라인
# 명세서 v4 pipeline_order 준수:
# 연속신호 로드 → g통일 → 6kHz LPF(AAF겸용) → polyphase resample(5/12)
# → gain → noise → ±16g clip → INT16 양자화
# ★ LPF/리샘플은 4초 전체 연속신호에 1회. 윈도우별 필터 절대 금지.
# ============================================================

_sos_lpf = butter(8, LPF_CUTOFF, btype='low', fs=FS_IN, output='sos')

def bearing_to_class(bearing: str) -> int:
    """베어링 코드 → 4클래스 라벨"""
    if bearing.startswith('K0'): return 0  # Normal
    if bearing.startswith('KA'): return 1  # OuterRace
    if bearing.startswith('KI'): return 2  # InnerRace
    if bearing.startswith('KB'): return 3  # Combined
    raise ValueError(f'알 수 없는 베어링 코드: {bearing}')

def parse_filename(fname: str):
    """N09_M07_F10_K001_1.mat 형식 파서"""
    stem  = os.path.splitext(os.path.basename(fname))[0]
    parts = stem.split('_')
    if len(parts) < 5:
        return None
    speed, load, force, bearing, trial = parts[0], parts[1], parts[2], parts[3], parts[4]
    try:
        label = bearing_to_class(bearing)
    except ValueError:
        return None
    return {
        'speed': speed, 'load': load, 'force': force,
        'bearing': bearing, 'trial': trial,
        'cond': f'{speed}_{load}_{force}',
        'label': label,
    }

def load_vibration(path: str) -> np.ndarray:
    """Paderborn MAT에서 vibration_1 채널 추출 (연속 float64)"""
    mat  = loadmat(path, squeeze_me=True, struct_as_record=False)
    keys = [k for k in mat.keys() if not k.startswith('__')]
    top  = mat[keys[0]]
    for ch in np.atleast_1d(top.Y):
        name = str(getattr(ch, 'Name', '') or getattr(ch, 'name', '')).lower()
        if 'vibration_1' in name:
            return np.asarray(ch.Data, dtype=np.float64).ravel()
    raise ValueError(f'vibration_1 채널 없음: {path}')

def convert_to_iis3dwb(sig_g: np.ndarray, randomize: bool = False,
                        seed: int = None) -> np.ndarray:
    """
    연속 g신호 → 연속 INT16 (IIS3DWB 도메인)
    pipeline_order: LPF → resample → gain → noise → clip → INT16
    ★ 윈도우 생성은 이 함수 이후에. 여기서는 절대 자르지 않음.
    """
    rng = np.random.default_rng(seed)
    x   = sig_g.astype(np.float64)

    # [1] 6kHz LPF — 전체 연속신호에 1회 (AAF 겸용)
    x = sosfiltfilt(_sos_lpf, x)

    # [2] Polyphase 리샘플 64000 → 80000/3 Hz
    x = resample_poly(x, RESAMPLE_UP, RESAMPLE_DOWN)

    # [3] Train-only 도메인 랜덤화
    if randomize:
        x = x * rng.uniform(0.95, 1.05)                   # gain variation
        x = x + rng.normal(0.0, NOISE_RMS_G, size=x.shape)  # band-limited noise
    else:
        x = x + rng.normal(0.0, NOISE_RMS_G, size=x.shape)  # val/test도 노이즈 고정 적용

    # [4] ±16g 클리핑
    x = np.clip(x, -FULL_SCALE_G, FULL_SCALE_G)

    # [5] INT16 양자화 (0.488mg/LSB)
    counts = np.round(x / SENSITIVITY).astype(np.int64)
    counts = np.clip(counts, -32768, 32767).astype(np.int16)
    return counts

print('공통 함수 정의 완료')
print(f'resample 비율: {RESAMPLE_UP}/{RESAMPLE_DOWN} = {RESAMPLE_UP/RESAMPLE_DOWN:.6f}')
print(f'예상 출력 주파수: {FS_IN * RESAMPLE_UP / RESAMPLE_DOWN:.4f} Hz')

공통 함수 정의 완료
resample 비율: 5/12 = 0.416667
예상 출력 주파수: 26666.6667 Hz


In [5]:
# ============================================================
# [공통 함수] 계열별 층화 분할 (KA/KI/KB/K0 각 6:2:2)
# ============================================================

def split_by_bearing_stratified(records):
    """
    KA/KI/KB/K0 계열별로 독립적으로 6:2:2 분할
    → 모든 클래스가 train/val/test 세 split에 균형 있게 분포
    → 동일 베어링은 반드시 한 split에만 (누수 방지)
    """
    bearings    = sorted(set(r['bearing'] for r in records))
    groups      = {'K0': [], 'KA': [], 'KI': [], 'KB': []}
    for b in bearings:
        for prefix in groups:
            if b.startswith(prefix):
                groups[prefix].append(b)
                break

    print('\n[SPLIT] 계열별 베어링 수:')
    for g, blist in groups.items():
        print(f'  {g}: {len(blist)}종 → {blist}')

    # 계열별 최소 3개 보장 검증
    for g, blist in groups.items():
        assert len(blist) >= 3, f'[{g}] 계열이 3개 미만 ({blist}) — 분할 불가'

    rng = np.random.default_rng(SEED)
    split = {'train': [], 'val': [], 'test': []}

    for g, blist in groups.items():
        b = list(blist)
        rng.shuffle(b)
        n    = len(b)
        n_tr = max(1, int(round(n * SPLIT_RATIO['train'])))
        n_va = max(1, int(round(n * SPLIT_RATIO['val'])))
        n_te = n - n_tr - n_va
        if n_te < 1:
            n_te = 1
            n_tr = n - n_va - n_te
        tr, va, te = b[:n_tr], b[n_tr:n_tr+n_va], b[n_tr+n_va:]
        split['train'] += tr
        split['val']   += va
        split['test']  += te
        print(f'  [{g}] train={tr}  val={va}  test={te}')

    split = {k: set(v) for k, v in split.items()}

    # 누수 검증
    all_b = list(split['train']) + list(split['val']) + list(split['test'])
    assert len(all_b) == len(set(all_b)), '★ 누수 발생! 베어링이 여러 split에 중복됨'

    # 각 split에 4클래스 모두 존재 검증
    for sp_name, sp_set in split.items():
        classes_in = set(bearing_to_class(b) for b in sp_set)
        missing    = set(range(N_CLASSES)) - classes_in
        assert not missing, f'[{sp_name}] split에 클래스 {missing} 누락 (베어링: {sorted(sp_set)})'

    print('\n[SPLIT] 최종 분할 결과:')
    for sp_name, sp_set in split.items():
        cls_cnt = {}
        for b in sp_set:
            cn = CLASS_NAMES[bearing_to_class(b)]
            cls_cnt[cn] = cls_cnt.get(cn, 0) + 1
        print(f'  {sp_name:5s}: {sorted(sp_set)}')
        print(f'         클래스분포: {cls_cnt}')

    return split

print('분할 함수 정의 완료')

분할 함수 정의 완료


---
## 셀 1: MVP 검증 (베어링당 2파일)
전체 2560개 돌리기 전에, 소량으로 파이프라인 전체를 검증합니다.
- 파일명에 `cls`가 포함되어 있는지 확인
- 도메인 변환 정상 동작 확인
- manifest.json 구조 확인

In [6]:
# ============================================================
# 셀 1. MVP 파이프라인 검증 (베어링당 2파일 제한)
# ============================================================

MVP_MODE               = True   # ★ MVP: True / 전체실행: False
MVP_MAX_PER_BEARING    = 2

# ── 기존 OUT_DIR 초기화 (이전 이진분류 npy 삭제) ──
import shutil
if os.path.exists(OUT_DIR):
    shutil.rmtree(OUT_DIR)
    print(f'[초기화] 기존 폴더 삭제: {OUT_DIR}')
os.makedirs(OUT_DIR, exist_ok=True)

# ── 파일 스캔 ──
if not os.path.isdir(MAT_DIR):
    raise FileNotFoundError(f'MAT_DIR 없음: {MAT_DIR}\n→ 경로를 확인하세요')

mat_files = sorted(glob.glob(os.path.join(MAT_DIR, '**', '*.mat'), recursive=True))
records   = []
for path in mat_files:
    meta = parse_filename(path)
    if meta is None:
        continue
    meta['path']  = path
    meta['fname'] = os.path.basename(path)
    records.append(meta)

bearings = sorted(set(r['bearing'] for r in records))
normal   = [b for b in bearings if b.startswith('K0')]
fault    = [b for b in bearings if not b.startswith('K0')]
print(f'[SCAN] 총 파일: {len(records)}개')
print(f'[SCAN] 정상({len(normal)}): {normal}')
print(f'[SCAN] 결함({len(fault)}): {fault}')
print(f'[SCAN] 운전조건: {sorted(set(r["cond"] for r in records))}')

# ── 베어링 분할 ──
split = split_by_bearing_stratified(records)

# ── MVP 제한 적용 ──
if MVP_MODE:
    by_bearing = {}
    kept       = []
    for r in records:
        by_bearing.setdefault(r['bearing'], 0)
        if by_bearing[r['bearing']] < MVP_MAX_PER_BEARING:
            by_bearing[r['bearing']] += 1
            kept.append(r)
    records = kept
    print(f'\n[MVP] 베어링당 최대 {MVP_MAX_PER_BEARING}개 → 처리 대상 {len(records)}개')

# ── 변환 & 저장 ──
def which_split(bearing):
    for sp_name, sp_set in split.items():
        if bearing in sp_set:
            return sp_name
    return None

manifest = []
n_ok, n_fail = 0, 0

for i, r in enumerate(records):
    sp = which_split(r['bearing'])
    if sp is None:
        continue
    try:
        sig   = load_vibration(r['path'])
        is_tr = (sp == 'train')
        fseed = SEED + abs(hash(r['fname'])) % 100000
        q     = convert_to_iis3dwb(sig, randomize=is_tr, seed=fseed)

        # ★ 파일명에 cls 포함 (4클래스 버전 식별용)
        out_name = f"{sp}__cls{r['label']}__{r['bearing']}__{r['cond']}__{r['trial']}.npy"
        np.save(os.path.join(OUT_DIR, out_name), q)

        manifest.append({
            'split':      sp,
            'bearing':    r['bearing'],
            'label':      r['label'],
            'class_name': CLASS_NAMES[r['label']],
            'cond':       r['cond'],
            'trial':      r['trial'],
            'n_samples':  int(q.shape[0]),
            'out':        out_name,
        })
        n_ok += 1
        if (i + 1) % 20 == 0:
            print(f'  진행 {i+1}/{len(records)}')
    except Exception as e:
        n_fail += 1
        print(f'  [FAIL] {r["fname"]}: {e}')

# manifest 저장
with open(os.path.join(OUT_DIR, 'manifest.json'), 'w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print(f'\n{'='*55}')
print(f'[DONE] 변환 성공 {n_ok}개 / 실패 {n_fail}개')
print(f'[DONE] fs_out = {FS_OUT:.4f} Hz')
print(f'[DONE] 저장: {OUT_DIR}')
print(f'{'='*55}')

# ── split별 처리 결과 요약 ──
print('\n[SUMMARY] split별 클래스 분포:')
for sp in ['train', 'val', 'test']:
    cnt = {}
    for m in manifest:
        if m['split'] == sp:
            cnt[m['class_name']] = cnt.get(m['class_name'], 0) + 1
    print(f'  {sp:5s}: {cnt}')

# ── cls 파일명 확인 ──
print('\n[파일명 확인] 처음 5개:')
npy_files = sorted(glob.glob(os.path.join(OUT_DIR, '*.npy')))
for f in npy_files[:5]:
    arr = np.load(f)
    print(f'  {os.path.basename(f)}')
    print(f'    shape={arr.shape}, dtype={arr.dtype}, min={arr.min()}, max={arr.max()}')

[초기화] 기존 폴더 삭제: /content/drive/My Drive/Colab Notebooks/Paderborn_iis3dwb
[SCAN] 총 파일: 2560개
[SCAN] 정상(6): ['K001', 'K002', 'K003', 'K004', 'K005', 'K006']
[SCAN] 결함(26): ['KA01', 'KA03', 'KA04', 'KA05', 'KA06', 'KA07', 'KA08', 'KA09', 'KA15', 'KA16', 'KA22', 'KA30', 'KB23', 'KB24', 'KB27', 'KI01', 'KI03', 'KI04', 'KI05', 'KI07', 'KI08', 'KI14', 'KI16', 'KI17', 'KI18', 'KI21']
[SCAN] 운전조건: ['N09_M07_F10', 'N15_M01_F10', 'N15_M07_F04', 'N15_M07_F10']

[SPLIT] 계열별 베어링 수:
  K0: 6종 → ['K001', 'K002', 'K003', 'K004', 'K005', 'K006']
  KA: 12종 → ['KA01', 'KA03', 'KA04', 'KA05', 'KA06', 'KA07', 'KA08', 'KA09', 'KA15', 'KA16', 'KA22', 'KA30']
  KI: 11종 → ['KI01', 'KI03', 'KI04', 'KI05', 'KI07', 'KI08', 'KI14', 'KI16', 'KI17', 'KI18', 'KI21']
  KB: 3종 → ['KB23', 'KB24', 'KB27']
  [K0] train=['K004', 'K003', 'K006', 'K005']  val=['K002']  test=['K001']
  [KA] train=['KA06', 'KA15', 'KA04', 'KA08', 'KA22', 'KA16', 'KA09']  val=['KA05', 'KA01']  test=['KA03', 'KA07', 'KA30']
  [KI] train=['KI21', 

---
## 셀 2: 전체 실행 (2560개)
셀 1에서 파일명에 `cls` 확인 + 변환 결과 정상이면 아래 셀을 실행합니다.

**MVP_MODE = False 로 바꾸고 재실행** — 약 20~40분 소요 예상

In [8]:
# ============================================================
# 셀 2. 전체 2560개 변환 (MVP_MODE=False)
# ============================================================

# 이후 셀에서 manifest를 공통으로 사용할 수 있도록 로드
def which_split_fn(bearing, sp_dict):
    for sp_name, sp_set in sp_dict.items():
        if bearing in sp_set:
            return sp_name
    return None

# ★ MVP 검증 완료 후 아래를 False로 변경
MVP_MODE = False

if not MVP_MODE:
    print('★ 전체 모드: 2560개 파일 변환 시작')
    print('  예상 소요 시간: 20~40분 (Colab T4 기준)')

    # OUT_DIR 재초기화
    if os.path.exists(OUT_DIR):
        shutil.rmtree(OUT_DIR)
    os.makedirs(OUT_DIR, exist_ok=True)

    # 파일 스캔 (전체)
    mat_files = sorted(glob.glob(os.path.join(MAT_DIR, '**', '*.mat'), recursive=True))
    records_all = []
    for path in mat_files:
        meta = parse_filename(path)
        if meta is None:
            continue
        meta['path']  = path
        meta['fname'] = os.path.basename(path)
        records_all.append(meta)

    print(f'[SCAN] 총 파일: {len(records_all)}개')

    # 분할 (전체 베어링 기준)
    split_full = split_by_bearing_stratified(records_all)

    manifest_full = []
    n_ok, n_fail  = 0, 0

    for i, r in enumerate(records_all):
        sp = which_split_fn(r['bearing'], split_full)
        if sp is None:
            continue
        try:
            sig   = load_vibration(r['path'])
            is_tr = (sp == 'train')
            fseed = SEED + abs(hash(r['fname'])) % 100000
            q     = convert_to_iis3dwb(sig, randomize=is_tr, seed=fseed)

            out_name = f"{sp}__cls{r['label']}__{r['bearing']}__{r['cond']}__{r['trial']}.npy"
            np.save(os.path.join(OUT_DIR, out_name), q)

            manifest_full.append({
                'split':      sp,
                'bearing':    r['bearing'],
                'label':      r['label'],
                'class_name': CLASS_NAMES[r['label']],
                'cond':       r['cond'],
                'trial':      r['trial'],
                'n_samples':  int(q.shape[0]),
                'out':        out_name,
            })
            n_ok += 1
            if (i + 1) % 100 == 0:
                print(f'  진행 {i+1}/{len(records_all)} ({n_ok}성공/{n_fail}실패)')
        except Exception as e:
            n_fail += 1
            print(f'  [FAIL] {r["fname"]}: {e}')

    with open(os.path.join(OUT_DIR, 'manifest.json'), 'w', encoding='utf-8') as f:
        json.dump(manifest_full, f, ensure_ascii=False, indent=2)
    manifest = manifest_full  # 이후 셀에서 사용

    print(f'\n[DONE] 성공 {n_ok}개 / 실패 {n_fail}개')
    print('[SUMMARY] split별 클래스 분포:')
    for sp in ['train', 'val', 'test']:
        cnt = {}
        for m in manifest_full:
            if m['split'] == sp:
                cnt[m['class_name']] = cnt.get(m['class_name'], 0) + 1
        print(f'  {sp:5s}: {cnt}')
else:
    print('MVP_MODE=True — 셀 1 결과를 사용합니다.')
    print('cls 파일명 확인 후 MVP_MODE=False로 변경하여 재실행하세요.')


★ 전체 모드: 2560개 파일 변환 시작
  예상 소요 시간: 20~40분 (Colab T4 기준)
[SCAN] 총 파일: 2560개

[SPLIT] 계열별 베어링 수:
  K0: 6종 → ['K001', 'K002', 'K003', 'K004', 'K005', 'K006']
  KA: 12종 → ['KA01', 'KA03', 'KA04', 'KA05', 'KA06', 'KA07', 'KA08', 'KA09', 'KA15', 'KA16', 'KA22', 'KA30']
  KI: 11종 → ['KI01', 'KI03', 'KI04', 'KI05', 'KI07', 'KI08', 'KI14', 'KI16', 'KI17', 'KI18', 'KI21']
  KB: 3종 → ['KB23', 'KB24', 'KB27']
  [K0] train=['K004', 'K003', 'K006', 'K005']  val=['K002']  test=['K001']
  [KA] train=['KA06', 'KA15', 'KA04', 'KA08', 'KA22', 'KA16', 'KA09']  val=['KA05', 'KA01']  test=['KA03', 'KA07', 'KA30']
  [KI] train=['KI21', 'KI01', 'KI08', 'KI07', 'KI17', 'KI18', 'KI16']  val=['KI04', 'KI05']  test=['KI14', 'KI03']
  [KB] train=['KB24']  val=['KB27']  test=['KB23']

[SPLIT] 최종 분할 결과:
  train: ['K003', 'K004', 'K005', 'K006', 'KA04', 'KA06', 'KA08', 'KA09', 'KA15', 'KA16', 'KA22', 'KB24', 'KI01', 'KI07', 'KI08', 'KI16', 'KI17', 'KI18', 'KI21']
         클래스분포: {'InnerRace': 7, 'OuterRace': 7, 'Nor

In [9]:
# ============================================================
# 셀 3. 윈도우 생성 (03_build_windows)
# manifest.json + npy → (N, 2048, 1) float32 학습 텐서
#
# 처리 순서:
# npy 로드 → 2048샘플 슬라이싱 → window mean subtraction(펌웨어 동일)
# → Train-only Aref 산출 → symmetric scaling [-1,1]
# ============================================================

SCALING_METHOD = 'symmetric'  # 권장: 'symmetric' or 'minmax'

# manifest 로드
manifest_path = os.path.join(OUT_DIR, 'manifest.json')
assert os.path.isfile(manifest_path), f'manifest.json 없음 — 셀 1/2를 먼저 실행하세요'
with open(manifest_path, encoding='utf-8') as f:
    manifest = json.load(f)

# split별 파일 그룹
splits_files = {'train': [], 'val': [], 'test': []}
missing = 0
for m in manifest:
    npy_path = os.path.join(OUT_DIR, m['out'])
    if not os.path.isfile(npy_path):
        missing += 1
        continue
    splits_files[m['split']].append({
        'counts_int16': np.load(npy_path),
        'label':        int(m['label']),
        'bearing':      m['bearing'],
        'cond':         m['cond'],
        'trial':        m.get('trial', ''),
        'source':       m['out'],
    })

print(f'manifest {len(manifest)}개 로드 / npy 누락 {missing}개')
for sp in ['train', 'val', 'test']:
    print(f'  {sp:5s}: 파일 {len(splits_files[sp])}개')

def slice_windows(counts_int16, is_train=True):
    """연속 INT16 → 윈도우 목록 + window mean subtraction"""
    x   = counts_int16.astype(np.float64)
    n   = len(x)
    hop = TRAIN_HOP if is_train else EVAL_HOP
    starts = list(range(0, n - WINDOW + 1, hop))
    if not starts:
        return np.empty((0, WINDOW))
    wins = np.empty((len(starts), WINDOW), dtype=np.float64)
    for i, s in enumerate(starts):
        seg       = x[s:s + WINDOW]
        seg       = seg - np.mean(seg)   # ★ window mean subtraction (DC 제거, 펌웨어 일치)
        wins[i]   = seg
    return wins

def build_windows(file_list, is_train=True):
    all_wins, all_y = [], []
    for fr in file_list:
        wins = slice_windows(fr['counts_int16'], is_train)
        if len(wins) == 0:
            continue
        all_wins.append(wins)
        all_y.extend([fr['label']] * len(wins))
    if not all_wins:
        return np.empty((0, WINDOW)), np.empty((0,), np.int32)
    return np.concatenate(all_wins, axis=0), np.array(all_y, dtype=np.int32)

# 윈도우 생성
print('\n윈도우 생성 중...')
tr_wins, tr_y   = build_windows(splits_files['train'], is_train=True)
va_wins, va_y   = build_windows(splits_files['val'],   is_train=False)
te_wins, te_y   = build_windows(splits_files['test'],  is_train=False)
print(f'  Train 윈도우: {tr_wins.shape}')
print(f'  Val   윈도우: {va_wins.shape}')
print(f'  Test  윈도우: {te_wins.shape}')

# ★ Train-only scaling (Aref = train 99.9 percentile)
AREF = float(np.percentile(np.abs(tr_wins), 99.9))
if AREF <= 0:
    AREF = 1.0
print(f'\n[Scaling] Aref (Train 99.9pct, count 도메인) = {AREF:.2f}')

def apply_scaling(wins):
    x = np.clip(wins / AREF, -1.0, 1.0).astype(np.float32)
    return x[..., np.newaxis]  # (N, W, 1)

X_tr  = apply_scaling(tr_wins)
X_val = apply_scaling(va_wins)
X_te  = apply_scaling(te_wins)
y_tr, y_val, y_te = tr_y, va_y, te_y

del tr_wins, va_wins, te_wins

# 클래스 분포 출력
def dist_str(y):
    u, c = np.unique(y, return_counts=True)
    return {CLASS_NAMES.get(int(k), int(k)): int(v) for k, v in zip(u, c)}

print(f'\n{'='*60}')
print(f'윈도우 데이터셋 생성 완료 (window={WINDOW}, scaling=symmetric)')
print(f'  Train : {X_tr.shape}  {dist_str(y_tr)}')
print(f'  Val   : {X_val.shape}  {dist_str(y_val)}')
print(f'  Test  : {X_te.shape}  {dist_str(y_te)}')
print(f'  범위  : [{X_tr.min():.4f}, {X_tr.max():.4f}]')
print(f'{'='*60}')

# 클래스 가중치 (Combined 부족 보정)
cw_arr       = compute_class_weight('balanced', classes=np.arange(N_CLASSES), y=y_tr)
class_weight = dict(enumerate(cw_arr))
print(f'class_weight: { {CLASS_NAMES[k]: round(v,2) for k, v in class_weight.items()} }')

manifest 2559개 로드 / npy 누락 0개
  train: 파일 1519개
  val  : 파일 480개
  test : 파일 560개

윈도우 생성 중...
  Train 윈도우: (156779, 2048)
  Val   윈도우: (25020, 2048)
  Test  윈도우: (29210, 2048)

[Scaling] Aref (Train 99.9pct, count 도메인) = 4547.13

윈도우 데이터셋 생성 완료 (window=2048, scaling=symmetric)
  Train : (156779, 2048, 1)  {'Normal': 33003, 'OuterRace': 57659, 'InnerRace': 57866, 'Combined': 8251}
  Val   : (25020, 2048, 1)  {'Normal': 4172, 'OuterRace': 8336, 'InnerRace': 8335, 'Combined': 4177}
  Test  : (29210, 2048, 1)  {'Normal': 4162, 'OuterRace': 12540, 'InnerRace': 8338, 'Combined': 4170}
  범위  : [-1.0000, 1.0000]
class_weight: {'Normal': np.float64(1.19), 'OuterRace': np.float64(0.68), 'InnerRace': np.float64(0.68), 'Combined': np.float64(4.75)}


In [10]:
# ============================================================
# 셀 4. 4클래스 1D Residual CNN 모델 정의 & 학습
# 명세서 v4 model 섹션:
# Conv1D(32, k=64, s=8) → ResBlk(64) → ResBlk(128) → ResBlk(128)
# → GAP → Dense(64) → Dropout → Dense(32) → Dense(4, softmax)
# ============================================================
from tensorflow.keras import layers, models, callbacks

def residual_block(x, filters, kernel_size=3):
    skip = x
    x    = layers.Conv1D(filters, kernel_size, padding='same')(x)
    x    = layers.BatchNormalization()(x)
    x    = layers.Activation('relu')(x)
    x    = layers.Conv1D(filters, kernel_size, padding='same')(x)
    x    = layers.BatchNormalization()(x)
    if skip.shape[-1] != filters:
        skip = layers.Conv1D(filters, 1, padding='same')(skip)
    x    = layers.Add()([x, skip])
    x    = layers.Activation('relu')(x)
    return x

def build_model_4class(window=2048, n_classes=4):
    """
    Wide-kernel 1D Residual CNN (명세서 v4 model 섹션)
    Neural-ART NPU 친화 구조 (지원 op만 사용)
    """
    inp = layers.Input(shape=(window, 1), name='vibration_input')

    # ★ Wide first kernel: 저주파 결함 충격 패턴 포착
    x = layers.Conv1D(32, 64, strides=8, padding='same', name='stem_conv')(inp)  # 2048 → 256
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling1D(2)(x)                                                  # 256 → 128

    x = residual_block(x, 64)      # 128
    x = layers.MaxPooling1D(2)(x)  # 64

    x = residual_block(x, 128)     # 64
    x = layers.MaxPooling1D(2)(x)  # 32

    x = residual_block(x, 128)     # 32

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(0.20)(x)

    # 4클래스 softmax 출력
    out = layers.Dense(n_classes, activation='softmax', name='output')(x)

    model = models.Model(inp, out, name='AI_KimBanjang_IIS3DWB_v4')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(3e-4),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_model_4class(WINDOW, N_CLASSES)
model.summary()
print(f'\n입력: {WINDOW}샘플 = {WINDOW/FS_OUT*1000:.1f}ms @ {FS_OUT:.0f}Hz (IIS3DWB)')
total_params = model.count_params()
print(f'파라미터 수: {total_params:,} ({total_params*4/1024:.1f} KB float32)')

Model: "AI_KimBanjang_IIS3DWB_v4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ vibration_input     │ (None, 2048, 1)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv1D)  │ (None, 256, 32)   │      2,080 │ vibration_input[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 256, 32)   │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 256, 32)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 128, 32)   │          0 │ activation[0][0]  │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 128, 64)   │      6,208 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 64)   │        256 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 128, 64)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 128, 64)   │     12,352 │ activation_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 64)   │        256 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 128, 64)   │      2,112 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 128, 64)   │          0 │ batch_normalizat… │
│                     │                   │            │ conv1d_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 128, 64)   │          0 │ add[0][0]         │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 64, 64)    │          0 │ activation_2[0][… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 64, 128)   │     24,704 │ max_pooling1d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 128)   │        512 │ conv1d_3[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 64, 128)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 64, 128)   │     49,280 │ activation_3[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 128)   │        512 │ conv1d_4[0][0]  

 Total params: 216,772 (846.77 KB)

 Trainable params: 215,428 (841.52 KB)

 Non-trainable params: 1,344 (5.25 KB)


입력: 2048샘플 = 76.8ms @ 26667Hz (IIS3DWB)
파라미터 수: 216,772 (846.8 KB float32)


In [11]:
# ── 학습 실행 ──
os.makedirs(SAVE_DIR, exist_ok=True)

cb_list = [
    callbacks.EarlyStopping(
        monitor='val_loss', patience=20,
        restore_best_weights=True, verbose=1,
        start_from_epoch=10
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=7, min_lr=1e-6, verbose=1
    ),
    callbacks.ModelCheckpoint(
        os.path.join(SAVE_DIR, 'best_iis3dwb_w2048.keras'),
        monitor='val_loss', save_best_only=True, verbose=0
    )
]

print(f'학습 시작: Train {X_tr.shape[0]}개, Val {X_val.shape[0]}개')
print(f'class_weight: {class_weight}')

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=128,
    class_weight=class_weight,
    callbacks=cb_list,
    verbose=1
)
print('\n학습 완료')

학습 시작: Train 156779개, Val 25020개
class_weight: {0: np.float64(1.1876117322667636), 1: np.float64(0.6797681194609688), 2: np.float64(0.6773364324473784), 3: np.float64(4.750302993576536)}
Epoch 1/100
1225/1225 ━━━━━━━━━━━━━━━━━━━━ 39s 19ms/step - accuracy: 0.9110 - loss: 0.1831 - val_accuracy: 0.2699 - val_loss: 4.9470 - learning_rate: 3.0000e-04
Epoch 2/100
1225/1225 ━━━━━━━━━━━━━━━━━━━━ 12s 10ms/step - accuracy: 0.9867 - loss: 0.0340 - val_accuracy: 0.2490 - val_loss: 7.0187 - learning_rate: 3.0000e-04
Epoch 3/100
1225/1225 ━━━━━━━━━━━━━━━━━━━━ 21s 10ms/step - accuracy: 0.9918 - loss: 0.0211 - val_accuracy: 0.3260 - val_loss: 5.2499 - learning_rate: 3.0000e-04
Epoch 4/100
1225/1225 ━━━━━━━━━━━━━━━━━━━━ 13s 11ms/step - accuracy: 0.9935 - loss: 0.0162 - val_accuracy: 0.2134 - val_loss: 6.7828 - learning_rate: 3.0000e-04
Epoch 5/100
1225/1225 ━━━━━━━━━━━━━━━━━━━━ 12s 10ms/step - accuracy: 0.9955 - loss: 0.0127 - val_accuracy: 0.3004 - val_loss: 6.5369 - learning_rate: 3.0000e-04
Epoch 6/

In [ ]:
# ── 학습 곡선 ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['loss'],     label='train_loss')
axes[0].plot(history.history['val_loss'], label='val_loss')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True)
axes[1].plot(history.history['accuracy'],     label='train_acc')
axes[1].plot(history.history['val_accuracy'], label='val_acc')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(True)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'training_curve_v4.png'), dpi=150)
plt.show()
print(f'best val_loss: {min(history.history["val_loss"]):.4f}')
print(f'best val_acc : {max(history.history["val_accuracy"])*100:.2f}%')

In [ ]:
# ============================================================
# 셀 5. Threshold 탐색 (Val 전용 — Test 누수 금지)
# 4클래스 softmax → 최대 확률 클래스 = argmax
# threshold: Normal 클래스 확률 < thr 이면 결함 판정
#
# 체크포인트:
#   Val FAR < 1%  (정상을 결함으로 오인)
#   Val FNR < 2%  (결함을 정상으로 놓침)
# ============================================================
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score

prob_val = model.predict(X_val, batch_size=128, verbose=0)  # (N, 4)

# argmax 기반 4클래스 평가
pred_val_argmax = np.argmax(prob_val, axis=1)

# 이진화(정상 vs 결함) 위한 Normal 확률 기반 threshold 탐색
normal_prob_val = prob_val[:, 0]  # Normal 클래스 확률

print('\n[4클래스 Val 평가 — argmax]')
val_acc  = accuracy_score(y_val, pred_val_argmax)
val_mf1  = f1_score(y_val, pred_val_argmax, average='macro', zero_division=0)
val_cm   = confusion_matrix(y_val, pred_val_argmax, labels=list(range(N_CLASSES)))
print(f'  Accuracy : {val_acc*100:.2f}%')
print(f'  Macro F1 : {val_mf1*100:.2f}%')
print(f'  Confusion Matrix (rows=true, cols=pred):')
print(f'  클래스: {[CLASS_NAMES[i] for i in range(N_CLASSES)]}')
print(val_cm)
print()
print(classification_report(y_val, pred_val_argmax,
                             target_names=[CLASS_NAMES[i] for i in range(N_CLASSES)],
                             zero_division=0))

# 이진화 threshold 탐색 (예지보전 KPI: FAR < 1%, FNR < 2%)
print('\n[Val threshold 탐색 — Normal확률 기준]')
print(f'{"thr":>5}  {"FNR%":>7}  {"FAR%":>7}  {"Acc%":>7}  {"MacroF1":>9}  KPI')
print('-' * 58)

best_thr, best_f1 = 0.5, 0.0
y_val_bin = (y_val > 0).astype(int)  # 0=Normal, 1=Fault(any)

for thr in np.arange(0.05, 0.96, 0.01):
    pred_bin = (normal_prob_val < thr).astype(int)  # Normal확률 < thr → 결함
    cm = confusion_matrix(y_val_bin, pred_bin, labels=[0, 1])
    if cm.shape != (2, 2):
        continue
    tn, fp, fn, tp = cm.ravel()
    fnr = fn/(fn+tp)*100 if (fn+tp) else 0
    far = fp/(fp+tn)*100 if (fp+tn) else 0
    acc = (tn+tp)/cm.sum()*100
    mf1 = f1_score(y_val_bin, pred_bin, average='macro', zero_division=0)*100
    ok  = '← PASS' if (fnr < 2.0 and far < 1.0) else ''
    if abs(thr*100 % 5) < 0.6:  # 0.05 간격으로만 출력
        print(f'{thr:>5.2f}  {fnr:>7.2f}  {far:>7.2f}  {acc:>7.2f}  {mf1:>9.2f}  {ok}')
    if fnr < 2.0 and far < 1.0 and mf1 > best_f1:
        best_f1, best_thr = mf1, round(float(thr), 3)

print(f'\n[Val 권장 threshold: {best_thr}  (Macro F1={best_f1:.2f}%)]')
if best_f1 == 0.0:
    print('★ KPI 만족 threshold 없음 — FAR/FNR 조건 미달. 학습 결과 재검토 필요.')
    best_thr = 0.5
    print(f'  → 기본값 0.5로 Test 평가 진행 (참고용)')
else:
    print(f'  → 이 threshold로 Test 평가 진행')

In [ ]:
# ============================================================
# 셀 6. Test 전체 평가
# ============================================================
import shutil

prob_te         = model.predict(X_te, batch_size=128, verbose=0)   # (N, 4)
pred_te_argmax  = np.argmax(prob_te, axis=1)                        # 4클래스 예측
normal_prob_te  = prob_te[:, 0]
pred_te_bin     = (normal_prob_te < best_thr).astype(int)           # 이진 예측
y_te_bin        = (y_te > 0).astype(int)

# 4클래스 KPI
te_acc4  = accuracy_score(y_te, pred_te_argmax)
te_mf14  = f1_score(y_te, pred_te_argmax, average='macro', zero_division=0)
te_cm4   = confusion_matrix(y_te, pred_te_argmax, labels=list(range(N_CLASSES)))

# 이진 KPI (예지보전 기준)
te_cm2   = confusion_matrix(y_te_bin, pred_te_bin, labels=[0, 1])
tn, fp, fn, tp = te_cm2.ravel()
te_fnr   = fn/(fn+tp)*100 if (fn+tp) else 0
te_far   = fp/(fp+tn)*100 if (fp+tn) else 0
te_acc2  = (tn+tp)/te_cm2.sum()*100
te_mf12  = f1_score(y_te_bin, pred_te_bin, average='macro', zero_division=0)*100

kpi_ok   = te_acc4 >= 0.98 and te_fnr < 2.0 and te_far < 1.0

print('=' * 65)
print(f'  AI 김반장 IIS3DWB v4 — Test 최종 KPI (thr={best_thr})')
print('=' * 65)
print(f'  [4클래스]')
print(f'  Accuracy  : {te_acc4*100:6.2f}%   (목표 ≥ 98%)')
print(f'  Macro F1  : {te_mf14*100:6.2f}%   (목표 ≥ 97%)')
print(f'  [예지보전 이진 KPI]')
print(f'  FNR (미탐): {te_fnr:6.2f}%   (목표 < 2%)')
print(f'  FAR (오경보): {te_far:5.2f}%   (목표 < 1%)')
print(f'  판정: {"✅ KPI 달성" if kpi_ok else "❌ KPI 미달 (정직 보고)"}')
print('=' * 65)
print()
print(classification_report(y_te, pred_te_argmax,
                             target_names=[CLASS_NAMES[i] for i in range(N_CLASSES)],
                             zero_division=0))

# 4클래스 Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
class_labels = [CLASS_NAMES[i] for i in range(N_CLASSES)]
sns.heatmap(te_cm4, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels, ax=axes[0])
axes[0].set_title(f'Test CM — 4클래스 (Acc={te_acc4*100:.1f}%)')
axes[0].set_ylabel('True'); axes[0].set_xlabel('Pred')
sns.heatmap(te_cm2, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Normal','Fault'], yticklabels=['Normal','Fault'], ax=axes[1])
axes[1].set_title(f'Test CM — 이진 (FNR={te_fnr:.1f}%, FAR={te_far:.1f}%)')
axes[1].set_ylabel('True'); axes[1].set_xlabel('Pred')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'confusion_matrix_v4.png'), dpi=150)
plt.show()

# 베어링별 상세 분석
print('\n[Test 베어링별 상세]')
print(f'{"베어링":8} {"클래스":12} {"샘플":6} {"Acc%":8} {"F1%":8}')
print('-' * 46)
# manifest에서 test 베어링 목록 재구성
te_manifest = [m for m in manifest if m['split'] == 'test']
te_bearings = sorted(set(m['bearing'] for m in te_manifest))
# 베어링별 마스크 (순서 일치 확인 필요 — 윈도우 순서와 동일)
bearing_labels_te = []
for fr in splits_files['test']:
    wins = slice_windows(fr['counts_int16'], is_train=False)
    bearing_labels_te.extend([(fr['bearing'], fr['label'])] * len(wins))
bl_arr = np.array(bearing_labels_te)
for b in te_bearings:
    mask  = bl_arr[:, 0] == b
    true_c = y_te[mask]
    pred_c = pred_te_argmax[mask]
    if len(true_c) == 0: continue
    b_acc = accuracy_score(true_c, pred_c)
    b_f1  = f1_score(true_c, pred_c, average='macro', zero_division=0)
    cls   = CLASS_NAMES.get(int(true_c[0]), '?')
    print(f'{b:8} {cls:12} {len(true_c):6} {b_acc*100:8.2f} {b_f1*100:8.2f}')

In [ ]:
# ============================================================
# 셀 7. INT8 양자화 & 검증
# 명세서 v4 quantization 섹션:
#   Float/INT8 Argmax 일치율 ≥ 99.5%
#   Accuracy 감소 ≤ 0.5%p, FNR 증가 ≤ 0.5%p, FAR 증가 ≤ 0.5%p
# ============================================================

# representative dataset (1000~2000, 클래스 균형)
def representative_dataset_gen():
    n_per_class = 400  # 4클래스 × 400 = 1600
    for c in range(N_CLASSES):
        idx = np.where(y_tr == c)[0]
        chosen = np.random.choice(idx, min(n_per_class, len(idx)), replace=False)
        for i in chosen:
            yield [X_tr[i:i+1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations              = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset    = representative_dataset_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type      = tf.int8
converter.inference_output_type     = tf.int8

print('INT8 양자화 변환 중...')
tflite_model = converter.convert()

TFLITE_PATH = os.path.join(SAVE_DIR, 'iis3dwb_w2048_int8.tflite')
with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)
print(f'저장: {TFLITE_PATH}  ({len(tflite_model)/1024:.1f} KB)')

# 인터프리터 설정
interp     = tf.lite.Interpreter(model_content=tflite_model)
interp.allocate_tensors()
inp_det    = interp.get_input_details()[0]
out_det    = interp.get_output_details()[0]
INP_SCALE  = inp_det['quantization'][0]
INP_ZP     = inp_det['quantization'][1]
OUT_SCALE  = out_det['quantization'][0]
OUT_ZP     = out_det['quantization'][1]
print(f'Input  scale={INP_SCALE:.8f}, zero_point={INP_ZP}')
print(f'Output scale={OUT_SCALE:.8f}, zero_point={OUT_ZP}')

# TFLite 검증 (Test 전체)
print(f'\nTFLite 검증: Test {len(X_te)}개...')
tflite_preds = []
for i in range(len(X_te)):
    x_q = np.clip(np.round(X_te[i:i+1] / INP_SCALE + INP_ZP), -128, 127).astype(np.int8)
    interp.set_tensor(inp_det['index'], x_q)
    interp.invoke()
    out     = interp.get_tensor(out_det['index']).ravel().astype(np.float32)
    out_f32 = (out - OUT_ZP) * OUT_SCALE
    tflite_preds.append(int(np.argmax(out_f32)))
    if (i + 1) % 1000 == 0:
        print(f'  {i+1}/{len(X_te)}...')

tflite_preds   = np.array(tflite_preds)
match_rate     = np.mean(tflite_preds == pred_te_argmax) * 100
tflite_acc4    = accuracy_score(y_te, tflite_preds) * 100
tflite_mf1     = f1_score(y_te, tflite_preds, average='macro', zero_division=0) * 100
tflite_pred_bin= (1 - (tflite_preds == 0)).astype(int)
tflite_cm2     = confusion_matrix(y_te_bin, tflite_pred_bin, labels=[0, 1])
tn2, fp2, fn2, tp2 = tflite_cm2.ravel()
tflite_fnr     = fn2/(fn2+tp2)*100 if (fn2+tp2) else 0
tflite_far     = fp2/(fp2+tn2)*100 if (fp2+tn2) else 0

print(f'\n{'='*60}')
print(f'  INT8 양자화 검증 결과')
print(f'{'='*60}')
print(f'  Argmax 일치율 : {match_rate:.2f}%  (목표 ≥ 99.5%)')
print(f'  INT8 Accuracy : {tflite_acc4:.2f}%  (Float: {te_acc4*100:.2f}%, 손실: {te_acc4*100-tflite_acc4:.2f}%p)')
print(f'  INT8 Macro F1 : {tflite_mf1:.2f}%  (Float: {te_mf14*100:.2f}%)')
print(f'  INT8 FNR      : {tflite_fnr:.2f}%  (Float: {te_fnr:.2f}%, 증가: {tflite_fnr-te_fnr:.2f}%p)')
print(f'  INT8 FAR      : {tflite_far:.2f}%  (Float: {te_far:.2f}%, 증가: {tflite_far-te_far:.2f}%p)')
ptq_ok = (match_rate >= 99.5 and
           abs(te_acc4*100 - tflite_acc4) <= 0.5 and
           tflite_fnr - te_fnr <= 0.5 and
           tflite_far - te_far <= 0.5)
print(f'  PTQ 판정      : {"✅ PASS" if ptq_ok else "❌ FAIL → QAT 검토 필요"}')
print(f'{'='*60}')

In [ ]:
# ============================================================
# 셀 8. 펌웨어 헤더 생성 & Drive 저장
# iis3dwb_constants.h (명세서 v4 firmware_header 섹션)
# ============================================================

# threshold quantization: Q_thr = round(thr / OUT_SCALE) + OUT_ZP
Q_THR = int(round(best_thr / OUT_SCALE)) + OUT_ZP

# fixed-point multiplier/shift 계산 (INT16 count → INT8)
# x_int8 = clip(round(x_count / AREF * 127), -128, 127)
# = round(x_count * (127/AREF))
# fixed-point: multiplier = round(127/AREF * 2^shift)
SHIFT = 15
FIXEDPOINT_MULT = int(round(127.0 / AREF * (2 ** SHIFT)))

header_content = f"""/**
 * AI 김반장 LITE — IIS3DWB 모델 상수 v4 (자동 생성)
 * 생성 스크립트: AI_KimBanjang_Paderborn_IIS3DWB_v4.ipynb
 * 모델: iis3dwb_w2048_int8.tflite
 * 클래스: 0=Normal / 1=OuterRace / 2=InnerRace / 3=Combined
 *
 * KPI:
 *   Float Accuracy : {te_acc4*100:.2f}%  FNR: {te_fnr:.2f}%  FAR: {te_far:.2f}%
 *   INT8  Accuracy : {tflite_acc4:.2f}%  Match: {match_rate:.2f}%
 */
#ifndef IIS3DWB_AI_CONSTANTS_H
#define IIS3DWB_AI_CONSTANTS_H

/* 센서 파라미터 */
#define AI_SAMPLE_RATE_NUMERATOR    80000U        /* IIS3DWB ODR = 80000/3 Hz */
#define AI_SAMPLE_RATE_DENOMINATOR  3U
#define IIS3DWB_SENSITIVITY_G_PER_LSB  0.000488f  /* ±16g, 0.488mg/LSB */
#define IIS3DWB_FULL_SCALE_G        16.0f

/* 윈도우 */
#define AI_WINDOW_SIZE              {WINDOW}U      /* 76.8ms @ 26666.6667Hz */

/* 전처리 (DC 제거 = window mean subtraction) */
#define AI_DC_REMOVAL_METHOD        WINDOW_MEAN_SUBTRACTION

/* 스케일링 (symmetric 99.9pct, Train-only) */
#define AI_AREF_COUNT               {AREF:.2f}f   /* Train abs(count) 99.9pct */
#define AI_FIXEDPOINT_MULTIPLIER    {FIXEDPOINT_MULT}       /* round(127/Aref * 2^{SHIFT}) */
#define AI_FIXEDPOINT_SHIFT         {SHIFT}U

/* TFLite INT8 양자화 파라미터 */
#define AI_INPUT_SCALE              {INP_SCALE:.8f}f
#define AI_INPUT_ZERO_POINT         ({int(INP_ZP)})
#define AI_OUTPUT_SCALE             {OUT_SCALE:.8f}f
#define AI_OUTPUT_ZERO_POINT        ({int(OUT_ZP)})

/* 예지보전 threshold */
#define AI_FAULT_THRESHOLD_FLOAT    {best_thr:.3f}f   /* Normal 확률 < 이 값이면 결함 */
#define AI_FAULT_THRESHOLD_Q        ({Q_THR})          /* quantized threshold */

/* 클래스 인덱스 */
#define AI_CLASS_NORMAL             0
#define AI_CLASS_OUTER_RACE         1
#define AI_CLASS_INNER_RACE         2
#define AI_CLASS_COMBINED           3
#define AI_NUM_CLASSES              {N_CLASSES}U

#endif /* IIS3DWB_AI_CONSTANTS_H */
"""

header_path = os.path.join(SAVE_DIR, 'iis3dwb_constants.h')
with open(header_path, 'w', encoding='utf-8') as f:
    f.write(header_content)
print(f'펌웨어 헤더 생성: {header_path}')
print()
print(header_content)

# Drive 저장 목록 확인
print('\n[Drive 저장 파일 목록]')
for fname in sorted(os.listdir(SAVE_DIR)):
    fpath = os.path.join(SAVE_DIR, fname)
    size  = os.path.getsize(fpath) / 1024
    print(f'  {fname:50s}  {size:.1f} KB')
print(f'\n저장 위치: {SAVE_DIR}')
print('\n✅ 모든 작업 완료')